# graphql

> Query Cloudflare's GraphQL Analytics API with schema-aware chaining, batched requests and raw GraphQL

In [ ]:
#| default_exp graphql

In [ ]:
import httpx2
from fastcore.utils import *
from fastspec.gql import GqlSpec, GqlClient, GqlError, INTROSPECT
from fastcflare.core import _cf_auth

In [ ]:
_all_ = ['GqlError']

In [ ]:
from datetime import datetime, timezone, timedelta
from fastcore.test import *
from pyskills import xdir, doc

Cloudflare exposes R2 storage and operations, Workers, KV, D1, DNS, HTTP, and firewall analytics exclusively through its GraphQL Analytics API.

`CfGql` wraps [fastspec's GraphQL client](https://answerdotai.github.io/fastspec/gql.html) with Cloudflare's distilled schema and the same `CLOUDFLARE_*` credentials as `CloudflareApi`. Explore datasets offline with `xdir`, `doc`, and `gql.t`, chain attributes with keyword arguments to build field-checked queries, or pass raw GraphQL text directly.

## The distilled schema

`build_gql_spec` introspects Cloudflare's GraphQL endpoint and writes compact lookup tables to `fastcflare/gql_spec.py`. `CfGql` loads those tables offline with no startup request.

In [ ]:
CF_GQL_URL = 'https://api.cloudflare.com/client/v4/graphql'

def build_gql_spec(
    nm='fastcflare/gql_spec.py', # Output module path
    url=CF_GQL_URL, # Cloudflare GraphQL endpoint
    token=None, # API token; defaults to `CLOUDFLARE_API_TOKEN` when `api_key` and `email` aren't given
    email=None, # Account email for a global API key; defaults to `CLOUDFLARE_EMAIL`
    api_key=None, # Global API key; defaults to `CLOUDFLARE_API_KEY` when no token is given or set
):
    "Regenerate the distilled GraphQL schema tables shipped as `fastcflare.gql_spec`."
    hdrs, *_ = _cf_auth(token, None, email, api_key)
    r = httpx2.post(url, json=dict(query=INTROSPECT), headers=hdrs, timeout=60)
    r.raise_for_status()
    GqlSpec.from_introspection(r.json()).save(nm)

In [ ]:
from fastcflare.gql_spec import gqlspec


In [ ]:
len(gqlspec['types']), gqlspec['query']

## CfGql

`CfGql` configures `GqlClient` with Cloudflare's GraphQL endpoint, the bundled schema, and `CLOUDFLARE_*` authentication. `account()` and `zone()` build filtered `viewer.accounts` and `viewer.zones` fragments.

In [ ]:
class CfGql(GqlClient):
    "Cloudflare GraphQL Analytics client using the shipped distilled schema"
    def __init__(self,
        token=None, # API token; defaults to `CLOUDFLARE_API_TOKEN` when `api_key` and `email` aren't given
        account_id=None, # Account ID; defaults to `CLOUDFLARE_ACCOUNT_ID`
        email=None, # Account email for a global API key; defaults to `CLOUDFLARE_EMAIL`
        api_key=None, # Global API key; defaults to `CLOUDFLARE_API_KEY` when no token is given or set
    ):
        from fastcflare.gql_spec import gqlspec
        hdrs, self.account_id, *_ = _cf_auth(token, account_id, email, api_key)
        super().__init__(GqlSpec.from_dict(gqlspec), CF_GQL_URL, headers=hdrs)

    def account(self, account_id=None):
        "Account fragment for `account_id` (defaults to `self.account_id`)"
        return self.viewer.accounts(filter={'accountTag': account_id or self.account_id})

    def zone(self, zone_id):
        "Zone fragment for `zone_id`"
        return self.viewer.zones(filter={'zoneTag': zone_id})

## Discovery and R2 analytics

Use `xdir` and `doc` to inspect available datasets and filter arguments offline without sending a request:

In [ ]:
api_key = os.getenv('CLOUDFLARE_API_KEY')
gql = CfGql(api_key=api_key) if api_key else CfGql()
xdir(gql.account(), 'r2')

In [ ]:
doc(gql.account().r2StorageAdaptiveGroups)

Inspect filter input objects and `orderBy` enums with `gql.t`:

In [ ]:
doc(gql.t.AccountR2StorageAdaptiveGroupsOrderBy)

Query per-bucket R2 storage usage over the last two days:

In [ ]:
if api_key and gql.account_id:
    since = (datetime.now(timezone.utc) - timedelta(days=2)).strftime('%Y-%m-%d')
    f = gql.account().r2StorageAdaptiveGroups(filter={'date_geq': since}, limit=20, orderBy=['max_payloadSize_DESC'])
    rows = (await f('dimensions { bucketName } max { payloadSize objectCount }'))[0]
    assert isinstance(rows, (list, L))
    [(r.dimensions.bucketName, r.max.payloadSize, r.max.objectCount) for r in rows]


## Validation and raw queries

Unfinished fragments and unknown fields fail locally before any network request:

In [ ]:
with expect_fail(TypeError, contains='needs a selection'): await gql.account()
with expect_fail(AttributeError, contains='has no field'): gql.account().no_such_dataset

## Export -

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()